# 05 · Workflows and agents

**Session 01 · notebook 5 of 6**

## Objective

Give the application two tools, document search and indicator data, and compare two ways of deciding which to call: a fixed workflow written in code, and a loop in which the model chooses. See how to bound the loop, inspect its decisions, and handle a failed call.

## The application after this notebook

```
                 ┌──> search_documents(query) ──> passages ──┐
question ──> model                                           ├──> model ──> answer
                 └──> get_indicator(indicator, country) ──> data ──┘
                 (fixed order in code, or chosen by the model, with a step limit and a trace)
```

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "pydantic", "scikit-learn", "requests"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook uses recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
llm = w.LLM()
print("Model client:", llm.mode())

Model client: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
import json

docs = w.load_documents()
chunks = [c for d in docs for c in w.chunk_sections(d)]
retriever = w.Retriever(chunks)


def search_documents(query, k=3):
    return [{"citation": h.chunk.citation(), "score": round(h.score, 3), "text": h.chunk.text} for h in retriever.search(query, k=k, recency_weight=0.2)]


handlers = {"search_documents": search_documents, "get_indicator": w.get_indicator}
tools = [w.SEARCH_TOOL, w.INDICATOR_TOOL]

INSTRUCTIONS = """You are a research assistant for development data.
Use get_indicator for every statistic and search_documents for definitions, context, and explanations.
Name the indicator and the source with each figure; cite passages in the square-bracket form given.
If the tools do not provide what is needed, say so. Do not quote figures from memory."""

## 1. A fixed workflow

The order of calls is written in code: search the documents, fetch the four series, compose. The model is called once, to write. Every run makes the same calls.

In [3]:
def fixed_workflow(question, scenario="nb05_compose"):
    passages = search_documents(question, k=4)
    series = [w.get_indicator(i, c, start=2010) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
    user = ("PASSAGES:\n" + "\n\n".join(f"{p['citation']}\n{p['text']}" for p in passages)
            + "\n\nDATA:\n" + json.dumps(series)
            + f"\n\nQUESTION: {question}")
    return llm.complete([{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": user}], scenario=scenario)


w.show(fixed_workflow(w.MAIN_QUESTION))

[recorded output: nb05_compose]
Internet use reached 35.0 percent of the population in Kenya and 31.7 percent in Rwanda in 2024
(World Development Indicators, IT.NET.USER.ZS). GDP per capita in current US dollars was 2,133 in
Kenya and 1,060 in Rwanda in the same year (NY.GDP.PCAP.CD). Since 2010, internet use has risen by
about 28 percentage points in Kenya and 24 in Rwanda; GDP per capita has roughly doubled in Kenya
and risen by about 80 percent in Rwanda. The retrieved briefs attribute Kenya's 2023 dip in dollar
GDP per capita to a currency depreciation [kenya-economy · GDP per capita · 2026] and note that
Rwanda's growth exceeded 7 percent in most years [rwanda-economy · Growth · 2026].


## 2. The model chooses the tools

The same tools, offered to the model with their schemas. The loop runs what it asks for, up to a step limit, and records a trace.

In [4]:
llm.reset()
final, trace = w.run_tool_loop(
    llm, [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": w.MAIN_QUESTION}],
    tools=tools, handlers=handlers, scenario="nb05_agent_main", max_steps=4,
)
print()
w.show(final)
print(f"\n{len(trace)} tool calls in {max(t['step'] for t in trace)} step(s)")

step 1: search_documents({"query": "internet use and GDP per capita trends Kenya Rwanda", "k": 4}) -> [{"citation": "[rwanda-economy \u00b7 intro \u00b7 2026]", "score": 0.596, "text": "Rwanda: GDP per capita and growth, 2010 to 2024\nintro\n# Rwanda: GDP per ca…
step 1: get_indicator({"indicator": "IT.NET.USER.ZS", "country": "Kenya", "start": 2010, "end": 2024}) -> {"indicator": "IT.NET.USER.ZS", "country": "KEN", "name": "Individuals using the Internet (% of population)", "unit": "", "source": "World Development Indicator…
step 1: get_indicator({"indicator": "IT.NET.USER.ZS", "country": "Rwanda", "start": 2010, "end": 2024}) -> {"indicator": "IT.NET.USER.ZS", "country": "RWA", "name": "Individuals using the Internet (% of population)", "unit": "", "source": "World Development Indicator…
step 1: get_indicator({"indicator": "NY.GDP.PCAP.CD", "country": "Kenya", "start": 2010, "end": 2024}) -> {"indicator": "NY.GDP.PCAP.CD", "country": "KEN", "name": "GDP per capita (current US$)", "un

The model chose the same calls the fixed workflow made, in one step. On a different question it would choose differently, which is the point and the risk.

## 3. A question outside the tools

The planted out-of-scope question. A good loop searches, finds nothing usable, and says so.

In [5]:
llm.reset()
final, trace = w.run_tool_loop(
    llm, [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": "How have Kenya's coffee export earnings changed in recent years?"}],
    tools=tools, handlers=handlers, scenario="nb05_agent_offscope", max_steps=3,
)
print()
w.show(final)

step 1: search_documents({"query": "Kenya coffee export earnings recent years", "k": 3}) -> [{"citation": "[coffee \u00b7 intro \u00b7 2026]", "score": 0.38, "text": "Coffee exports in East Africa\nintro\nCoffee is one of the traditional export crops o…

The available material does not support an answer with figures. The document collection has one note
on coffee in East Africa, which describes the sector in general terms and states that it carries no
figures [coffee · intro · 2026]. The data tool holds five indicators (internet use, mobile
subscriptions, GDP per capita, GDP growth, population) and no series on coffee exports. To answer
this question, a trade or agricultural statistics source would be needed.


## 4. A failed call

The model asks for an indicator id that does not exist. The tool returns an error object; the model reads it and retries with the right id. Returning errors as data is what lets the loop recover.

In [6]:
llm.reset()
final, trace = w.run_tool_loop(
    llm, [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": "What share of Rwanda's population used the internet in 2023?"}],
    tools=tools, handlers=handlers, scenario="nb05_agent_error", max_steps=4,
)
print()
w.show(final)

step 1: get_indicator({"indicator": "IT.NET.USERS.ZS", "country": "Rwanda", "start": 2023, "end": 2023}) -> {"error": "unknown indicator 'IT.NET.USERS.ZS'", "available": ["NY.GDP.PCAP.CD", "IT.NET.USER.ZS", "NY.GDP.MKTP.KD.ZG", "SP.POP.TOTL", "IT.CEL.SETS.P2"]}
step 2: get_indicator({"indicator": "IT.NET.USER.ZS", "country": "Rwanda", "start": 2023, "end": 2023}) -> {"indicator": "IT.NET.USER.ZS", "country": "RWA", "name": "Individuals using the Internet (% of population)", "unit": "", "source": "World Development Indicator…

In 2023, 28.8 percent of Rwanda's population used the internet (World Development Indicators,
IT.NET.USER.ZS).


## 5. Bounding the loop

A step limit is the simplest control. With `max_steps=1` the loop returns after the first round of tool calls and says that it stopped. An application decides what to do then: answer with what it has, ask the user, or fail.

In [7]:
llm.reset()
final, trace = w.run_tool_loop(
    llm, [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": w.MAIN_QUESTION}],
    tools=tools, handlers=handlers, scenario="nb05_agent_main", max_steps=1, verbose=False,
)
print(final)
print(len(trace), "tool calls were made before the limit")

(stopped: step limit reached without a final answer)
5 tool calls were made before the limit


## 6. Choosing between the two

| | Fixed workflow | Model-directed loop |
|---|---|---|
| Calls per run | Known in advance | Varies with the question |
| Behaviour on a new kind of question | Same calls, possibly irrelevant | Adapts, possibly wrongly |
| Cost | Predictable | Bounded by the step limit |
| Testing | Test the composer | Test the trace and the answer |
| Use when | The questions have one shape | The questions vary and the tools are few and safe |

The worked example has one shape, so the fixed workflow is the right design for it. The loop earns its place when users ask questions the designer did not anticipate, and only with a step limit, a trace, and tools that fail as data.

## Key points

- A workflow written in code is predictable and testable; prefer it when the task has one shape.
- A model-directed loop adapts to the question; bound it with a step limit, keep a trace, and return tool errors as data.
- Both need the same two things from the tools: good descriptions and results the model can read.

## Exercise

Remove `get_indicator` from `tools` and rerun section 2. Read what the model does without a data tool and whether the instructions stop it from quoting figures from memory. In recorded mode, write down the expected behaviour; with a key, compare.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.